In [5]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle,
    ListFlowable, ListItem, HRFlowable, KeepTogether, NextPageTemplate,
    PageTemplate, Frame, BaseDocTemplate, FrameBreak
)
from reportlab.platypus.flowables import Flowable
from reportlab.pdfgen import canvas as canvas_mod

In [6]:
# -*- coding: utf-8 -*-
"""
Builds a detailed, beginner-to-advanced friendly PDF study guide on
Greedy Algorithms & Intervals for DSA interview prep.
"""



# ----------------------------------------------------------------------
# COLORS
# ----------------------------------------------------------------------
NAVY = colors.HexColor("#1B2A4A")
BLUE = colors.HexColor("#2F5DA8")
LIGHT_BLUE_BG = colors.HexColor("#EAF1FB")
TIP_BG = colors.HexColor("#E8F6EE")
TIP_BORDER = colors.HexColor("#2E9E5B")
WARN_BG = colors.HexColor("#FDF1E3")
WARN_BORDER = colors.HexColor("#D98324")
CODE_BG = colors.HexColor("#1E1E2E")
CODE_FG = colors.HexColor("#E5E5E5")
CODE_ACCENT = colors.HexColor("#8BE9FD")
GREY_TEXT = colors.HexColor("#4A4A4A")
GOLD = colors.HexColor("#C9A227")
TABLE_HEADER_BG = NAVY
TABLE_ALT_BG = colors.HexColor("#F3F6FB")

PAGE_W, PAGE_H = A4

# ----------------------------------------------------------------------
# STYLES
# ----------------------------------------------------------------------
styles = getSampleStyleSheet()

styles.add(ParagraphStyle(
    name="CoverTitle", fontName="Helvetica-Bold", fontSize=34, leading=40,
    textColor=colors.white, alignment=TA_CENTER, spaceAfter=6
))
styles.add(ParagraphStyle(
    name="CoverSubtitle", fontName="Helvetica", fontSize=16, leading=22,
    textColor=colors.HexColor("#D7E3F7"), alignment=TA_CENTER, spaceAfter=4
))
styles.add(ParagraphStyle(
    name="CoverTag", fontName="Helvetica-Oblique", fontSize=11.5, leading=16,
    textColor=colors.HexColor("#AFC6ED"), alignment=TA_CENTER
))
styles.add(ParagraphStyle(
    name="H1", fontName="Helvetica-Bold", fontSize=20, leading=24,
    textColor=NAVY, spaceBefore=4, spaceAfter=10,
    borderPadding=0,
))
styles.add(ParagraphStyle(
    name="H2", fontName="Helvetica-Bold", fontSize=14.5, leading=18,
    textColor=BLUE, spaceBefore=14, spaceAfter=6,
))
styles.add(ParagraphStyle(
    name="H3", fontName="Helvetica-Bold", fontSize=12, leading=15,
    textColor=NAVY, spaceBefore=10, spaceAfter=4,
))
styles.add(ParagraphStyle(
    name="BodyText2", parent=styles["Normal"], fontName="Helvetica", fontSize=10.3,
    leading=15, textColor=colors.HexColor("#222222"), alignment=TA_JUSTIFY,
    spaceAfter=6,
))
styles.add(ParagraphStyle(
    name="BulletText", parent=styles["BodyText2"], leftIndent=0, spaceAfter=3,
))
styles.add(ParagraphStyle(
    name="CodeText", fontName="Courier", fontSize=8.6, leading=11.6,
    textColor=CODE_FG, alignment=TA_LEFT,
))
styles.add(ParagraphStyle(
    name="CodeComment", fontName="Courier-Oblique", fontSize=8.6, leading=11.6,
    textColor=colors.HexColor("#8B9AAE"), alignment=TA_LEFT,
))
styles.add(ParagraphStyle(
    name="TipText", parent=styles["BodyText2"], textColor=colors.HexColor("#1B4332"),
    spaceAfter=2,
))
styles.add(ParagraphStyle(
    name="WarnText", parent=styles["BodyText2"], textColor=colors.HexColor("#5C3A14"),
    spaceAfter=2,
))
styles.add(ParagraphStyle(
    name="BoxHeading", fontName="Helvetica-Bold", fontSize=10.5, leading=13,
    spaceAfter=3,
))
styles.add(ParagraphStyle(
    name="TOCEntry", fontName="Helvetica", fontSize=11, leading=20,
    textColor=colors.HexColor("#222222"),
))
styles.add(ParagraphStyle(
    name="TOCEntryBold", fontName="Helvetica-Bold", fontSize=12.5, leading=24,
    textColor=NAVY, spaceBefore=8,
))
styles.add(ParagraphStyle(
    name="FooterQuote", fontName="Helvetica-Oblique", fontSize=9.5,
    textColor=GREY_TEXT, alignment=TA_CENTER,
))
styles.add(ParagraphStyle(
    name="SmallCaption", fontName="Helvetica-Oblique", fontSize=8.7,
    textColor=colors.HexColor("#6B6B6B"), alignment=TA_LEFT, spaceBefore=2,
))

# ----------------------------------------------------------------------
# HELPER FLOWABLES / FUNCTIONS
# ----------------------------------------------------------------------

def P(text, style="BodyText2"):
    return Paragraph(text, styles[style])


def H1(text):
    return [Spacer(1, 4), P(text, "H1"),
            HRFlowable(width="100%", thickness=1.4, color=BLUE, spaceAfter=10)]


def H2(text, num=None):
    label = f"{text}"
    return [P(label, "H2")]


def H3(text):
    return [P(text, "H3")]


def bullets(items, style="BulletText", bullet_char="\u2022"):
    flow = []
    for it in items:
        flow.append(Paragraph(f"{bullet_char}&nbsp;&nbsp;{it}", styles[style]))
    return flow


def numbered(items, style="BulletText"):
    flow = []
    for i, it in enumerate(items, 1):
        flow.append(Paragraph(f"<b>{i}.</b>&nbsp;&nbsp;{it}", styles[style]))
    return flow


def code_block(code, title=None):
    """A dark-themed code box using a single-cell Table."""
    lines = code.strip("\n").split("\n")
    paras = []
    if title:
        paras.append(Paragraph(f'<font color="#8BE9FD"><b>{title}</b></font>',
                                ParagraphStyle(name="ct", fontName="Courier-Bold",
                                               fontSize=8.8, textColor=CODE_ACCENT,
                                               spaceAfter=4)))
    for ln in lines:
        ln_esc = (ln.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))
        ln_esc = ln_esc.replace(" ", "&nbsp;")
        style = "CodeComment" if ln.strip().startswith("#") else "CodeText"
        paras.append(Paragraph(ln_esc if ln_esc else "&nbsp;", styles[style]))
    tbl = Table([[paras]], colWidths=[162 * mm])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), CODE_BG),
        ("LEFTPADDING", (0, 0), (-1, -1), 10),
        ("RIGHTPADDING", (0, 0), (-1, -1), 10),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
        ("BOX", (0, 0), (-1, -1), 0.6, colors.HexColor("#33334D")),
        ("ROUNDEDCORNERS", [4, 4, 4, 4]),
    ]))
    return KeepTogether([tbl, Spacer(1, 8)])


def callout_box(text_list, kind="tip", heading=None):
    """kind: 'tip' or 'warn'. text_list is list of strings (paragraphs) or Paragraph objects."""
    if kind == "tip":
        bg, border, txt_style, default_heading, icon = TIP_BG, TIP_BORDER, "TipText", "TIP", "\u2713"
        heading_color = "#1B4332"
    else:
        bg, border, txt_style, default_heading, icon = WARN_BG, WARN_BORDER, "WarnText", "COMMON CONFUSION", "!"
        heading_color = "#5C3A14"
    heading_txt = heading or default_heading
    inner = [Paragraph(f'<font color="{heading_color}">{icon} <b>{heading_txt}</b></font>',
                        styles["BoxHeading"])]
    for t in text_list:
        if isinstance(t, str):
            inner.append(Paragraph(t, styles[txt_style]))
        else:
            inner.append(t)
    tbl = Table([[inner]], colWidths=[162 * mm])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), bg),
        ("LEFTPADDING", (0, 0), (-1, -1), 12),
        ("RIGHTPADDING", (0, 0), (-1, -1), 10),
        ("TOPPADDING", (0, 0), (-1, -1), 7),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
        ("LINEBEFORE", (0, 0), (0, -1), 3.2, border),
        ("BOX", (0, 0), (-1, -1), 0.4, border),
    ]))
    return KeepTogether([tbl, Spacer(1, 9)])


def info_table(data, col_widths=None, header=True, font_size=9.3):
    """data: list of rows (list of strings or Paragraph)."""
    wrapped = []
    for ri, row in enumerate(data):
        wrapped_row = []
        for ci, cell in enumerate(row):
            if isinstance(cell, str):
                st = "BodyText2"
                if ri == 0 and header:
                    cell = f'<font color="white"><b>{cell}</b></font>'
                wrapped_row.append(Paragraph(cell, ParagraphStyle(
                    name=f"tc{ri}{ci}", parent=styles["BodyText2"],
                    fontSize=font_size, leading=font_size + 3.2, alignment=TA_LEFT)))
            else:
                wrapped_row.append(cell)
        wrapped.append(wrapped_row)
    t = Table(wrapped, colWidths=col_widths, repeatRows=1 if header else 0)
    style_cmds = [
        ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#C9D4E3")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ]
    if header:
        style_cmds.append(("BACKGROUND", (0, 0), (-1, 0), TABLE_HEADER_BG))
    for r in range(1 if header else 0, len(wrapped)):
        if (r - (1 if header else 0)) % 2 == 1:
            style_cmds.append(("BACKGROUND", (0, r), (-1, r), TABLE_ALT_BG))
    t.setStyle(TableStyle(style_cmds))
    return t


def problem_header(num, title, difficulty, pattern):
    diff_color = {"Easy": "#2E9E5B", "Medium": "#D98324", "Hard": "#C0392B"}.get(difficulty, "#333333")
    tbl = Table([[
        Paragraph(f'<font color="white"><b>Q{num}.</b></font>', ParagraphStyle(
            name="qn", fontSize=12, textColor=colors.white)),
        Paragraph(f'<font color="white"><b>{title}</b></font>', ParagraphStyle(
            name="qt", fontSize=12.5, textColor=colors.white)),
        Paragraph(f'<font color="{diff_color}"><b>{difficulty}</b></font>', ParagraphStyle(
            name="qd", fontSize=10, alignment=TA_CENTER)),
    ]], colWidths=[18 * mm, 110 * mm, 34 * mm])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), NAVY),
        ("BACKGROUND", (2, 0), (2, 0), colors.white),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
        ("BOX", (0, 0), (-1, -1), 0.6, NAVY),
    ]))
    pattern_p = Paragraph(f'<i>Pattern: {pattern}</i>', styles["SmallCaption"])
    return [tbl, pattern_p, Spacer(1, 5)]


def section_label(text):
    return Paragraph(f'<font color="#2F5DA8"><b>{text}</b></font>',
                      ParagraphStyle(name="seclabel", fontSize=10.6, spaceBefore=6, spaceAfter=3))


# ----------------------------------------------------------------------
# PAGE DECORATION (header/footer + cover background)
# ----------------------------------------------------------------------
TITLE_TEXT = "Greedy Algorithms & Intervals"

def on_page(canvas, doc):
    canvas.saveState()
    if doc.page == 1:
        # Cover page full navy background
        canvas.setFillColor(NAVY)
        canvas.rect(0, 0, PAGE_W, PAGE_H, stroke=0, fill=1)
        canvas.setFillColor(GOLD)
        canvas.rect(0, PAGE_H - 10, PAGE_W, 4, stroke=0, fill=1)
        canvas.rect(0, 0, PAGE_W, 4, stroke=0, fill=1)
    else:
        # header bar
        canvas.setFillColor(NAVY)
        canvas.rect(0, PAGE_H - 16 * mm, PAGE_W, 16 * mm, stroke=0, fill=1)
        canvas.setFillColor(colors.white)
        canvas.setFont("Helvetica-Bold", 9.5)
        canvas.drawString(18 * mm, PAGE_H - 10.5 * mm, "DSA MASTER NOTES")
        canvas.setFont("Helvetica", 9)
        canvas.drawRightString(PAGE_W - 18 * mm, PAGE_H - 10.5 * mm, TITLE_TEXT)
        # footer
        canvas.setStrokeColor(colors.HexColor("#C9D4E3"))
        canvas.setLineWidth(0.6)
        canvas.line(18 * mm, 14 * mm, PAGE_W - 18 * mm, 14 * mm)
        canvas.setFillColor(GREY_TEXT)
        canvas.setFont("Helvetica", 8.3)
        canvas.drawString(18 * mm, 9.5 * mm, "Greedy Algorithms & Intervals \u2014 Complete Guide")
        canvas.drawRightString(PAGE_W - 18 * mm, 9.5 * mm, f"Page {doc.page - 1}")
    canvas.restoreState()


# ----------------------------------------------------------------------
# DOCUMENT
# ----------------------------------------------------------------------
doc = BaseDocTemplate(
    "Greedy_Algorithms_Intervals_DSA_Guide.pdf",
    pagesize=A4,
    leftMargin=18 * mm, rightMargin=18 * mm,
    topMargin=22 * mm, bottomMargin=16 * mm,
    title="Greedy Algorithms & Intervals - Complete DSA Guide",
    author="DSA Master Notes",
)
frame = Frame(doc.leftMargin, doc.bottomMargin, doc.width, doc.height, id="normal")
doc.addPageTemplates([PageTemplate(id="main", frames=[frame], onPage=on_page)])

story = []

# ======================================================================
# COVER PAGE
# ======================================================================
story.append(Spacer(1, 70 * mm))
story.append(P("GREEDY ALGORITHMS", "CoverTitle"))
story.append(P("&amp; INTERVALS", "CoverTitle"))
story.append(Spacer(1, 8))
story.append(P("The Complete Beginner-to-Advanced Guide", "CoverSubtitle"))
story.append(Spacer(1, 14))
story.append(HRFlowable(width="40%", thickness=1, color=GOLD, hAlign="CENTER", spaceAfter=14))
story.append(P("Most-asked interview questions &bull; Patterns &bull; Tips &amp; Tricks<br/>"
               "Confusion Points Explained &bull; Full Code Walkthroughs &bull; Complexity Analysis",
               "CoverTag"))
story.append(Spacer(1, 90 * mm))
story.append(P("DSA Master Notes Series", "CoverTag"))
story.append(PageBreak())

# ======================================================================
# TABLE OF CONTENTS
# ======================================================================
story += H1("Table of Contents")
toc_entries = [
    ("1.", "What Exactly Is a Greedy Algorithm?"),
    ("2.", "Greedy vs. Dynamic Programming vs. Divide &amp; Conquer"),
    ("3.", "How to Prove a Greedy Algorithm Is Correct"),
    ("4.", "Understanding Intervals \u2014 The Core Building Block"),
    ("5.", "The Golden Rule: Which Way Should You Sort?"),
    ("6.", "Most-Asked Interview Questions (Full Walkthroughs)"),
    ("", "&nbsp;&nbsp;Q1. Merge Intervals"),
    ("", "&nbsp;&nbsp;Q2. Insert Interval"),
    ("", "&nbsp;&nbsp;Q3. Non-overlapping Intervals (Erase Overlap Intervals)"),
    ("", "&nbsp;&nbsp;Q4. Meeting Rooms I"),
    ("", "&nbsp;&nbsp;Q5. Meeting Rooms II"),
    ("", "&nbsp;&nbsp;Q6. Minimum Number of Arrows to Burst Balloons"),
    ("", "&nbsp;&nbsp;Q7. Interval List Intersections"),
    ("", "&nbsp;&nbsp;Q8. Employee Free Time"),
    ("", "&nbsp;&nbsp;Q9. Partition Labels"),
    ("", "&nbsp;&nbsp;Q10. Minimum Platforms Needed"),
    ("", "&nbsp;&nbsp;Q11. Car Pooling"),
    ("", "&nbsp;&nbsp;Q12. Jump Game"),
    ("", "&nbsp;&nbsp;Q13. Jump Game II"),
    ("", "&nbsp;&nbsp;Q14. Gas Station"),
    ("", "&nbsp;&nbsp;Q15. Task Scheduler"),
    ("", "&nbsp;&nbsp;Q16. Queue Reconstruction by Height"),
    ("7.", "Master Technique Toolbox (Tips &amp; Tricks)"),
    ("8.", "Confusion Points \u2014 Clarified in Depth"),
    ("9.", "Complexity Cheat Sheet"),
    ("10.", "Pattern Recognition: How to Spot a Greedy-Interval Question"),
    ("11.", "Practice List &amp; Final Interview Checklist"),
]
toc_rows = []
for num, title in toc_entries:
    style = "TOCEntryBold" if num and "." in num and len(num) <= 3 else "TOCEntry"
    toc_rows.append([Paragraph(f"<b>{num}</b>" if num else "", styles["TOCEntry"]),
                      Paragraph(title, styles[style])])
toc_table = Table(toc_rows, colWidths=[10 * mm, 150 * mm])
toc_table.setStyle(TableStyle([
    ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ("TOPPADDING", (0, 0), (-1, -1), 1),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 1),
]))
story.append(toc_table)
story.append(PageBreak())

# ======================================================================
# SECTION 1: WHAT IS GREEDY
# ======================================================================
story += H1("1. What Exactly Is a Greedy Algorithm?")

story.append(P(
    "A <b>greedy algorithm</b> builds a solution step by step, and at every single step it makes the "
    "choice that looks <b>best right now</b> \u2014 without ever going back to reconsider an earlier choice. "
    "It never \"undoes\" a decision. It trusts that a series of locally optimal decisions will add up to a "
    "globally optimal (i.e. the best possible) solution.", "BodyText2"))

story.append(P(
    "Think of it like paying for something with the fewest coins possible: you keep grabbing the largest "
    "coin that doesn't overshoot the remaining amount, over and over, until you're done. You never say "
    "\"let me swap that earlier coin for two smaller ones instead.\" That one-directional, never-look-back "
    "behaviour <b>is</b> the greedy strategy.", "BodyText2"))

story += H3("The Greedy Recipe (how every greedy solution is built)")
story += numbered([
    "<b>Identify the decision being made at each step</b> (e.g. \"which interval do I keep?\", \"which room do I assign?\").",
    "<b>Define a greedy choice rule</b> \u2014 a simple criterion that picks one option over another "
    "(e.g. \"always pick the interval that finishes earliest\").",
    "<b>Sort or organise the input</b> so that the greedy rule can be applied in a single left-to-right (or right-to-left) pass.",
    "<b>Iterate once</b>, applying the rule and updating your running answer.",
    "<b>Prove (or at least sanity-check) that no better choice existed</b> at each step \u2014 this is the part beginners skip, and it's exactly where bugs hide.",
])

story.append(callout_box([
    "Greedy algorithms are almost always <b>O(n log n)</b> because the expensive part is the sort, "
    "and the pass after sorting is <b>O(n)</b>. If you ever see a greedy problem needing more than one "
    "sort + one pass, double check your approach \u2014 you may be overcomplicating it."
], kind="tip", heading="TIP \u2014 Spotting the Time Complexity Instantly"))

story += H3("Why Greedy Can Go Wrong (and why interviewers probe this)")
story.append(P(
    "Not every optimisation problem can be solved greedily. The classic counter-example is "
    "<b>coin change with arbitrary denominations</b>: with coins {1, 3, 4} and a target of 6, the greedy "
    "approach picks 4 then 1 then 1 (three coins), but the optimal answer is 3 + 3 (two coins). Greedy "
    "\"looked best\" at each step but still lost to the globally optimal answer. This is why interviewers "
    "love asking <i>\"why does greedy work here?\"</i> \u2014 they want to see that you understand greedy is "
    "not a universal hammer.", "BodyText2"))

story.append(callout_box([
    "A huge number of candidates apply a greedy rule that sounds reasonable, get it accepted on the "
    "visible test cases, and then fail on a hidden edge case because they never actually verified <i>why</i> "
    "the rule is safe. Always ask yourself: <i>\"Can I construct an input where this local choice backfires "
    "later?\"</i> If you can't, that's a good (informal) signal the greedy choice is safe."
], kind="warn", heading="COMMON CONFUSION \u2014 \u201cIt passed my examples, so it must be correct\u201d"))

story.append(PageBreak())

# ======================================================================
# SECTION 2: GREEDY vs DP vs D&C
# ======================================================================
story += H1("2. Greedy vs. Dynamic Programming vs. Divide &amp; Conquer")

story.append(P(
    "Beginners frequently confuse these three paradigms because all three break a problem into smaller "
    "pieces. The difference is <b>how much freedom you give yourself to reconsider past decisions.</b>",
    "BodyText2"))

cmp_table = info_table([
    ["Aspect", "Greedy", "Dynamic Programming", "Divide &amp; Conquer"],
    ["Core idea", "Pick the locally best option and never revisit it",
     "Try all valid options at each step but reuse previously computed sub-answers (memoisation/tabulation)",
     "Split into independent sub-problems, solve each, combine results"],
    ["Looks back?", "Never", "Implicitly \u2014 via stored sub-problem answers", "No (sub-problems are independent)"],
    ["Typical complexity", "O(n log n) or O(n)", "O(n\u00b2), O(n\u00d7k), etc.", "O(n log n) typically"],
    ["Needs proof of correctness?", "Yes \u2014 exchange argument / matroid / induction",
     "Not really \u2014 correctness comes from trying every state", "Usually straightforward"],
    ["Example", "Activity selection, Huffman coding, interval scheduling",
     "0/1 Knapsack, Longest Common Subsequence", "Merge Sort, Quick Sort"],
], col_widths=[33 * mm, 42 * mm, 50 * mm, 37 * mm])
story.append(cmp_table)
story.append(Spacer(1, 8))

story.append(P(
    "A very useful litmus test: <b>if an interval/activity-selection style problem can be solved greedily, "
    "the exact same problem can almost always also be solved with (slower) DP.</b> If you're unsure whether "
    "greedy will work, mentally write the DP recurrence first (\"best answer considering items up to i, "
    "either taking or skipping item i\"), then check whether the greedy rule always happens to pick the same "
    "branch DP would pick. If yes \u2014 greedy is safe and much faster.", "BodyText2"))

story.append(callout_box([
    "Many \"Can this be done greedily?\" interview questions are really testing whether you know this "
    "relationship: <b>DP is the safety net; greedy is the shortcut you take once you've proven the safety "
    "net always agrees with the shortcut.</b>"
], kind="tip"))

story.append(PageBreak())

# ======================================================================
# SECTION 3: PROVING GREEDY CORRECTNESS
# ======================================================================
story += H1("3. How to Prove a Greedy Algorithm Is Correct")

story.append(P(
    "You will almost never need a formal, rigorous proof in an interview, but knowing these two standard "
    "argument styles lets you explain <i>why</i> your greedy choice is safe \u2014 which is exactly what "
    "separates a strong answer from a lucky guess.", "BodyText2"))

story += H3("A) The Exchange Argument (most common, very intuitive)")
story.append(P(
    "Assume there's some optimal solution <i>O</i> that does <b>not</b> follow your greedy rule at some step. "
    "Show that you can always \"exchange\" the conflicting piece of <i>O</i> with the greedy choice <b>without "
    "making the solution worse</b>. If you can always perform this swap and the solution never gets worse "
    "(it might even get better, or stay the same), then the greedy choice is at least as good as any optimal "
    "solution \u2014 which means greedy itself must also be optimal.", "BodyText2"))

story.append(P(
    "<b>Worked mini-example (Activity Selection):</b> Suppose the optimal solution's first chosen interval "
    "finishes later than the interval that finishes earliest overall. Swap it out for the earliest-finishing "
    "interval instead. Because this new interval finishes no later, it can only leave <b>more</b> room (or "
    "equal room) for everything that comes after \u2014 so the rest of the optimal solution still fits. "
    "The total count of chosen intervals doesn't decrease. Hence \u201calways pick the interval that finishes "
    "earliest\u201d is a safe greedy rule.", "BodyText2"))

story += H3("B) The Matroid / Stays-Ahead Argument (more advanced, good to mention)")
story.append(P(
    "You show that after each greedy step, your partial solution is \"at least as good\" as any other "
    "algorithm's partial solution after the same number of steps, for every prefix length. If greedy always "
    "\"stays ahead\", it must finish ahead (or tied) at the very end too.", "BodyText2"))

story.append(callout_box([
    "In an interview, you almost never need to write this out formally. A single sentence like "
    "\u201cif I pick the earliest-finishing interval first, I'm never worse off than any other first choice, "
    "because an earlier finish time only ever frees up more room for the rest\u201d is usually enough to show "
    "the interviewer you understand <i>why</i>, not just <i>that</i>, greedy works."
], kind="tip", heading="TIP \u2014 What to actually say out loud in an interview"))

story.append(PageBreak())

# ======================================================================
# SECTION 4: UNDERSTANDING INTERVALS
# ======================================================================
story += H1("4. Understanding Intervals \u2014 The Core Building Block")

story.append(P(
    "An <b>interval</b> is simply a pair <font face=\"Courier\">[start, end]</font> representing a continuous "
    "range \u2014 time, numbers, positions, anything with a beginning and an end. Interval problems are "
    "popular in interviews because they combine <b>sorting</b>, <b>greedy choice</b>, and <b>careful "
    "boundary-condition handling</b> \u2014 three skills interviewers love testing together.", "BodyText2"))

story += H3("When do two intervals overlap?")
story.append(P(
    "Given <font face=\"Courier\">A = [a1, a2]</font> and <font face=\"Courier\">B = [b1, b2]</font>, they "
    "overlap if and only if:", "BodyText2"))
story.append(code_block("overlap = (a1 <= b2) and (b1 <= a2)"))
story.append(P(
    "This single line is the single most reused formula in the entire topic. It appears, in some disguised "
    "form, in almost every problem in this guide.", "BodyText2"))

story.append(callout_box([
    "Does <font face=\"Courier\">[1, 3]</font> and <font face=\"Courier\">[3, 5]</font> overlap? This is the "
    "single most common source of wrong answers in interval problems, and the answer <b>depends entirely on "
    "the problem statement</b>. If intervals represent <b>time ranges where touching endpoints count as free "
    "(e.g. meeting ends at 3, next starts at 3)</b>, they do <b>not</b> overlap \u2014 use strict "
    "<font face=\"Courier\">&lt;</font>. If intervals represent <b>closed numeric ranges where sharing a "
    "boundary point still counts as overlapping</b>, they <b>do</b> overlap \u2014 use "
    "<font face=\"Courier\">&lt;=</font>. <b>Always re-read the problem statement's examples to confirm "
    "which rule applies before writing a single line of code.</b>"
], kind="warn", heading="COMMON CONFUSION \u2014 Touching intervals: overlap or not?"))

story += H3("Representations You'll See")
story += bullets([
    "<font face=\"Courier\">[[1,3],[2,6],[8,10]]</font> \u2014 list of [start, end] pairs (most common).",
    "Separate <font face=\"Courier\">start[]</font> and <font face=\"Courier\">end[]</font> arrays (classic \u201cMinimum Platforms\u201d style).",
    "<font face=\"Courier\">(start, duration)</font> pairs \u2014 remember to convert to <font face=\"Courier\">end = start + duration</font> first!",
    "Events encoded as <font face=\"Courier\">(time, +1/-1)</font> \u2014 the \u201csweep line\u201d / \u201cdifference array\u201d encoding (explained in Section 7).",
])

story.append(PageBreak())

# ======================================================================
# SECTION 5: SORTING DIRECTION
# ======================================================================
story += H1("5. The Golden Rule: Which Way Should You Sort?")

story.append(P(
    "If there's one single idea that unlocks 90% of interval problems, it's this: <b>the correct sort key "
    "is determined by what question you are trying to answer, not by habit.</b> Beginners often default to "
    "\u201csort by start\u201d for everything \u2014 that's wrong more often than it's right.", "BodyText2"))

sort_table = info_table([
    ["Goal", "Sort By", "Why"],
    ["Merge all overlapping intervals into blocks", "Start time (ascending)",
     "You need to scan left to right and grow/close a \u201ccurrent block\u201d \u2014 only works if starts are ordered."],
    ["Pick the maximum number of non-overlapping intervals (activity selection)", "End time (ascending)",
     "Finishing earliest always leaves the most room for future choices \u2014 proven by the exchange argument in Section 3."],
    ["Find the minimum number of intervals to remove so none overlap", "End time (ascending)",
     "Equivalent to activity selection: keep as many as possible, remove the rest."],
    ["Find minimum meeting rooms / max overlap at any point", "Start time, with ends in a min-heap OR separate sorted start[] and end[] arrays",
     "You need to know, at every new start, how many previous meetings haven't ended yet."],
    ["Burst balloons with minimum arrows", "End time (ascending)",
     "Same shape as activity selection \u2014 you're grouping overlapping intervals under one arrow, choosing the tightest right boundary first."],
    ["Insert a new interval into an already-sorted list", "(Already sorted by start \u2014 no re-sort needed)",
     "You only need one linear pass: intervals strictly before, interval(s) overlapping, intervals strictly after."],
], col_widths=[58 * mm, 40 * mm, 64 * mm])
story.append(sort_table)
story.append(Spacer(1, 8))

story.append(callout_box([
    "Before writing any code, say this sentence out loud: <i>\u201cI am sorting by ___ because I need to know "
    "___ as I scan left to right.\u201d</i> If you can't finish that sentence confidently, you haven't picked "
    "the right sort key yet \u2014 stop and think again rather than guessing."
], kind="tip", heading="TIP \u2014 The one sentence that prevents 90% of wrong-sort bugs"))

story.append(callout_box([
    "Sorting by start time when you actually needed end time (or vice-versa) is the #1 silent bug in interval "
    "problems \u2014 your code still runs, it's still syntactically fine, it just quietly gives the wrong "
    "answer on certain inputs. There's no crash to warn you, so you must reason about the sort key "
    "deliberately, every single time."
], kind="warn", heading="COMMON CONFUSION \u2014 Wrong sort key = silently wrong answer"))

story.append(PageBreak())

# ======================================================================
# SECTION 6: PROBLEM WALKTHROUGHS
# ======================================================================
story += H1("6. Most-Asked Interview Questions \u2014 Full Walkthroughs")
story.append(P(
    "Each problem below follows the same structure so you can study efficiently: "
    "<b>Problem \u2192 Example \u2192 Thought Process \u2192 Why Greedy Works \u2192 Code \u2192 Dry Run \u2192 "
    "Complexity \u2192 Pitfalls.</b> Read every \u201cWhy Greedy Works\u201d section carefully \u2014 that's "
    "the part interviewers actually grade you on.", "BodyText2"))
story.append(Spacer(1, 6))

# ---------------- Q1 Merge Intervals ----------------
story += problem_header(1, "Merge Intervals", "Medium", "Sort by start, sweep & merge")
story.append(P(
    "Given a list of intervals, merge all overlapping intervals and return a list of the non-overlapping "
    "intervals that cover all the ranges in the input.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("Input: [[1,3],[2,6],[8,10],[15,18]] &rarr; Output: [[1,6],[8,10],[15,18]] "
               "&nbsp;&nbsp;(because [1,3] and [2,6] overlap and combine into [1,6])", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "If we sort intervals by start time, any interval that overlaps with our \u201ccurrent merged block\u201d "
    "must appear immediately after it in this sorted order (because its start is small enough to still be "
    "<= the current block's end). So a single left-to-right scan is enough: keep a \u201ccurrent block\u201d, "
    "and either extend it or close it and start a new one.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "Sorting by start guarantees we never \u201cmiss\u201d an overlap that comes later out of order \u2014 "
    "every interval that could possibly merge with the current block is seen before any interval that can't. "
    "There's no backtracking needed because once we move past an interval, no future interval can ever have "
    "a smaller start that would have overlapped with something we already closed.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def merge(intervals):
    if not intervals:
        return []
    intervals.sort(key=lambda x: x[0])   # sort by START
    merged = [intervals[0]]

    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:            # overlap -> extend
            merged[-1][1] = max(last_end, end)
        else:                            # no overlap -> new block
            merged.append([start, end])
    return merged""", title="merge_intervals.py"))
story.append(section_label("Dry Run"))
story.append(P(
    "Sorted input: [1,3], [2,6], [8,10], [15,18]. Start with merged=[[1,3]]. Next [2,6]: 2<=3 so extend "
    "&rarr; merged=[[1,6]]. Next [8,10]: 8>6 so append &rarr; merged=[[1,6],[8,10]]. Next [15,18]: 15>10 so "
    "append &rarr; final merged=[[1,6],[8,10],[15,18]].", "BodyText2"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n log n) for the sort + O(n) for the scan. Space: O(n) for the output (O(log n) "
               "extra for the sort itself).", "BodyText2"))
story.append(callout_box([
    "Using <font face=\"Courier\">start &lt;= last_end</font> (not strict <) merges touching intervals like "
    "[1,3] and [3,5] into [1,5]. Check the problem statement's examples to see whether touching counts as "
    "overlapping \u2014 this single operator is the most common reason \u201ccorrect-looking\u201d code fails "
    "one hidden test case."
], kind="warn"))
story.append(PageBreak())

# ---------------- Q2 Insert Interval ----------------
story += problem_header(2, "Insert Interval", "Medium", "Three-zone linear scan")
story.append(P(
    "You're given a list of <b>non-overlapping</b> intervals already sorted by start time, plus one new "
    "interval. Insert the new interval, merging if necessary, and return the resulting sorted, "
    "non-overlapping list.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("Input: intervals=[[1,3],[6,9]], newInterval=[2,5] &rarr; Output: [[1,5],[6,9]]", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "Because the input is already sorted, you don't need to sort again \u2014 that's the whole point of this "
    "variant. Split the work into exactly three zones scanned left to right: "
    "(1) intervals that end strictly before the new interval starts &rarr; copy as-is, "
    "(2) intervals that overlap the new interval &rarr; keep expanding the new interval to absorb them, "
    "(3) intervals that start strictly after the new (possibly expanded) interval ends &rarr; copy as-is.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "Because the list was pre-sorted and pre-merged (no overlaps existed before insertion), once you pass "
    "zone 1 you will hit a contiguous run of overlapping intervals (zone 2) and then a contiguous run of "
    "non-overlapping ones (zone 3) \u2014 the zones can never interleave. That's what allows a single clean "
    "O(n) pass instead of re-sorting everything.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def insert(intervals, newInterval):
    result = []
    i, n = 0, len(intervals)
    ns, ne = newInterval

    # Zone 1: intervals ending before new interval starts
    while i < n and intervals[i][1] < ns:
        result.append(intervals[i])
        i += 1

    # Zone 2: overlapping intervals -> merge into (ns, ne)
    while i < n and intervals[i][0] <= ne:
        ns = min(ns, intervals[i][0])
        ne = max(ne, intervals[i][1])
        i += 1
    result.append([ns, ne])

    # Zone 3: remaining intervals start after new interval ends
    while i < n:
        result.append(intervals[i])
        i += 1

    return result""", title="insert_interval.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n) \u2014 no sorting needed since input is pre-sorted. Space: O(n) for the output.", "BodyText2"))
story.append(callout_box([
    "The biggest beginner mistake here is sorting the whole list again \u201cjust to be safe\u201d, turning an "
    "O(n) problem into an unnecessary O(n log n) one. Trust the problem's guarantee that the input is already "
    "sorted and non-overlapping."
], kind="warn"))
story.append(PageBreak())

# ---------------- Q3 Non-overlapping Intervals ----------------
story += problem_header(3, "Non-overlapping Intervals", "Medium", "Classic Activity Selection (sort by end)")
story.append(P(
    "Given a list of intervals, find the <b>minimum number of intervals you must remove</b> so that the "
    "rest do not overlap.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("Input: [[1,2],[2,3],[3,4],[1,3]] &rarr; Output: 1 (remove [1,3]).", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "This is secretly the same problem as \u201cActivity Selection\u201d: find the <b>maximum number of "
    "intervals you can KEEP</b> such that none overlap, then the answer is "
    "<font face=\"Courier\">total \u2212 kept</font>. Sort by end time, greedily keep an interval whenever "
    "its start is &gt;= the end of the last interval you kept.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "This is a direct application of the exchange argument from Section 3: always keeping the "
    "earliest-finishing valid interval leaves the maximum possible room for everything after it, so it can "
    "never be a worse choice than keeping any later-finishing interval instead.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def eraseOverlapIntervals(intervals):
    if not intervals:
        return 0
    intervals.sort(key=lambda x: x[1])   # sort by END
    kept_end = intervals[0][1]
    kept_count = 1

    for start, end in intervals[1:]:
        if start >= kept_end:            # no overlap -> keep it
            kept_end = end
            kept_count += 1
        # else: overlaps -> skip it (this is the one we "remove")

    return len(intervals) - kept_count""", title="non_overlapping_intervals.py"))
story.append(section_label("Dry Run"))
story.append(P(
    "Sorted by end: [1,2],[2,3],[1,3],[3,4]. Start kept_end=2, count=1. [2,3]: 2&gt;=2 keep, kept_end=3, "
    "count=2. [1,3]: 1&lt;3 skip (overlap). [3,4]: 3&gt;=3 keep, kept_end=4, count=3. Total=4, kept=3, "
    "removed = 4-3 = 1.", "BodyText2"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n log n). Space: O(1) extra (ignoring sort space).", "BodyText2"))
story.append(callout_box([
    "Sorting by <b>start</b> instead of <b>end</b> is the #1 wrong submission for this exact problem on "
    "LeetCode. It can look like it works on small examples and then fail on larger ones \u2014 always sort by "
    "end for this pattern."
], kind="warn"))
story.append(PageBreak())

# ---------------- Q4 Meeting Rooms I ----------------
story += problem_header(4, "Meeting Rooms I", "Easy", "Overlap detection via sort")
story.append(P(
    "Given an array of meeting time intervals, determine if a person could attend <b>all</b> of them "
    "(i.e., no two meetings overlap).", "BodyText2"))
story.append(section_label("Example"))
story.append(P("Input: [[0,30],[5,10],[15,20]] &rarr; Output: false (([0,30] overlaps with both others).", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "Sort by start time. If any meeting starts before the previous meeting ends, there's a clash and the "
    "answer is false. Otherwise, true.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def canAttendMeetings(intervals):
    intervals.sort(key=lambda x: x[0])
    for i in range(1, len(intervals)):
        if intervals[i][0] < intervals[i - 1][1]:
            return False
    return True""", title="meeting_rooms_1.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n log n). Space: O(1) extra.", "BodyText2"))
story.append(callout_box([
    "This is the \u201cbaby version\u201d of Meeting Rooms II below. If you can solve this one instantly, you "
    "already understand the overlap check \u2014 the harder version just needs to <i>count</i> overlaps "
    "instead of just detecting one."
], kind="tip"))
story.append(PageBreak())

# ---------------- Q5 Meeting Rooms II ----------------
story += problem_header(5, "Meeting Rooms II", "Medium", "Min-Heap on end times / two-pointer on sorted starts & ends")
story.append(P(
    "Given an array of meeting time intervals, find the <b>minimum number of conference rooms</b> required "
    "so that all meetings can be held without clashing.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("Input: [[0,30],[5,10],[15,20]] &rarr; Output: 2 (meeting [0,30] overlaps with both [5,10] "
               "and [15,20], but [5,10] and [15,20] don't overlap each other, so 2 rooms suffice).", "BodyText2"))
story.append(section_label("Thought Process \u2014 Approach A (Min-Heap)"))
story.append(P(
    "Sort meetings by start time. Use a min-heap that stores the <b>end time</b> of every meeting currently "
    "using a room. For each new meeting, if the room that frees up <b>earliest</b> (heap top) ends at or "
    "before this meeting's start, reuse that room (pop and push the new end time). Otherwise, you need a "
    "brand new room (just push, heap grows). The heap's final size is the max rooms used at once.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "Always trying to reuse the room that frees up <b>soonest</b> is optimal: if that room can't be reused, "
    "then no other occupied room (which frees up even later) could be reused either \u2014 so you genuinely "
    "need a new room. This is the exchange argument again, applied to room-freeing times instead of interval "
    "ends.", "BodyText2"))
story.append(section_label("Code (Python) \u2014 Min-Heap Approach"))
story.append(code_block(
"""import heapq

def minMeetingRooms(intervals):
    if not intervals:
        return 0
    intervals.sort(key=lambda x: x[0])   # sort by START
    heap = []  # stores END times of rooms in use

    for start, end in intervals:
        if heap and heap[0] <= start:
            heapq.heapreplace(heap, end)  # reuse the earliest-freeing room
        else:
            heapq.heappush(heap, end)     # need a brand new room

    return len(heap)""", title="meeting_rooms_2_heap.py"))
story.append(section_label("Thought Process \u2014 Approach B (Two Sorted Arrays / Sweep Line)"))
story.append(P(
    "Split into a <font face=\"Courier\">starts[]</font> array and an <font face=\"Courier\">ends[]</font> "
    "array, sort both independently. Walk through with two pointers: every time a meeting starts, that's "
    "+1 room needed; every time a meeting ends (and it ends at or before the current start), that's \u22121 "
    "room freed. Track the running count and its maximum \u2014 that maximum is the answer.", "BodyText2"))
story.append(code_block(
"""def minMeetingRooms_v2(intervals):
    starts = sorted(i[0] for i in intervals)
    ends   = sorted(i[1] for i in intervals)

    rooms = 0
    max_rooms = 0
    s = e = 0
    n = len(intervals)

    while s < n:
        if starts[s] < ends[e]:
            rooms += 1              # a meeting starts, needs a room
            s += 1
        else:
            rooms -= 1              # a meeting ended, frees a room
            e += 1
        max_rooms = max(max_rooms, rooms)

    return max_rooms""", title="meeting_rooms_2_sweep.py"))
story.append(section_label("Complexity"))
story.append(P("Both approaches: Time O(n log n), Space O(n) for the heap or the two sorted arrays.", "BodyText2"))
story.append(callout_box([
    "This problem is really asking <b>\u201cwhat is the maximum number of intervals overlapping at any single "
    "instant?\u201d</b> \u2014 recognise that phrase and you'll immediately know to reach for the min-heap or "
    "sweep-line pattern, which reappears constantly (CPU scheduling, booking systems, \u201ccar pooling\u201d "
    "below, etc.)."
], kind="tip", heading="TIP \u2014 Recognising this pattern elsewhere"))
story.append(callout_box([
    "In the two-pointer version, using <font face=\"Courier\">starts[s] &lt; ends[e]</font> (strict) vs "
    "<font face=\"Courier\">&lt;=</font> changes whether a meeting that starts exactly when another ends "
    "needs a new room. Match this to whichever overlap rule Section 4 told you the problem wants."
], kind="warn"))
story.append(PageBreak())

# ---------------- Q6 Minimum Arrows ----------------
story += problem_header(6, "Min. Number of Arrows to Burst Balloons", "Medium", "Sort by end, greedy grouping")
story.append(P(
    "Balloons are represented as intervals <font face=\"Courier\">[x_start, x_end]</font> along a 2D plane. "
    "An arrow shot straight up at position <font face=\"Courier\">x</font> bursts every balloon whose "
    "interval contains <font face=\"Courier\">x</font>. Find the minimum number of arrows needed to burst "
    "all balloons.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("Input: [[10,16],[2,8],[1,6],[7,12]] &rarr; Output: 2 (one arrow at x=6 bursts [2,8] and "
               "[1,6]; one arrow at x=11 bursts [10,16] and [7,12]).", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "This is Non-overlapping Intervals (Q3) wearing a disguise! Sort by end position. Shoot the first arrow "
    "at the end of the first balloon. Every subsequent balloon whose start is &lt;= that arrow's position "
    "gets burst \u201cfor free\u201d by the same arrow. The moment you find a balloon whose start is beyond "
    "the current arrow's position, you need a new arrow \u2014 place it at that balloon's end.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "Placing the arrow at the <b>earliest possible end</b> among the current group of overlapping balloons "
    "maximises the chance that later balloons also get caught by the same shot \u2014 exactly the same "
    "exchange argument as activity selection, just phrased as \u201cballoons\u201d instead of \u201cintervals "
    "you keep\u201d.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def findMinArrowShots(points):
    if not points:
        return 0
    points.sort(key=lambda x: x[1])   # sort by END
    arrows = 1
    arrow_pos = points[0][1]

    for start, end in points[1:]:
        if start > arrow_pos:         # current arrow can't reach it
            arrows += 1
            arrow_pos = end
        # else: this balloon is already burst by the current arrow

    return arrows""", title="min_arrows_burst_balloons.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n log n). Space: O(1) extra.", "BodyText2"))
story.append(callout_box([
    "Here, touching endpoints (balloon A ends exactly where balloon B starts) count as <b>overlapping</b> "
    "(one arrow bursts both) \u2014 opposite convention from \u201cMeeting Rooms\u201d where touching is "
    "usually fine. Always verify with the problem's own examples instead of assuming the convention carries "
    "over from a similar-looking problem."
], kind="warn"))
story.append(PageBreak())

# ---------------- Q7 Interval List Intersections ----------------
story += problem_header(7, "Interval List Intersections", "Medium", "Two-pointer merge of two sorted lists")
story.append(P(
    "Given two lists of <b>disjoint, sorted</b> intervals, return their intersection \u2014 every range of "
    "time that is covered by <b>both</b> lists.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("A = [[0,2],[5,10],[13,23],[24,25]], B = [[1,5],[8,12],[15,24],[25,26]] "
               "&rarr; Output: [[1,2],[5,5],[8,10],[15,23],[24,24],[25,25]]", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "Use two pointers, one for each list. The intersection of the current pair, if any, is "
    "<font face=\"Courier\">[max(a_start, b_start), min(a_end, b_end)]</font> \u2014 keep it only if start "
    "&lt;= end. Then advance whichever interval ends first (it can no longer intersect anything new in the "
    "other list), since both lists are already individually non-overlapping and sorted.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "Advancing the pointer with the smaller end time is safe because that interval has \u201cexhausted its "
    "usefulness\u201d \u2014 every future interval in its own list starts even later, so it can never "
    "intersect anything the other pointer hasn't already been compared against.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def intervalIntersection(A, B):
    result = []
    i = j = 0
    while i < len(A) and j < len(B):
        lo = max(A[i][0], B[j][0])
        hi = min(A[i][1], B[j][1])
        if lo <= hi:
            result.append([lo, hi])

        if A[i][1] < B[j][1]:
            i += 1
        else:
            j += 1
    return result""", title="interval_list_intersections.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(m + n) \u2014 no sorting needed, inputs already sorted. Space: O(1) extra "
               "(excluding the output).", "BodyText2"))
story.append(callout_box([
    "No sorting step here because the problem guarantees both lists already come sorted and disjoint "
    "\u2014 don't waste time re-sorting."
], kind="tip"))
story.append(PageBreak())

# ---------------- Q8 Employee Free Time ----------------
story += problem_header(8, "Employee Free Time", "Hard", "Flatten + Merge Intervals")
story.append(P(
    "You're given a list of schedules, one per employee, each a list of non-overlapping intervals sorted by "
    "start time. Return the list of finite gaps in the schedule that are common to <b>all</b> employees "
    "(i.e. everyone is free during that time).", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "Flatten every employee's intervals into one big list (ignore which employee they belong to), run the "
    "exact <b>Merge Intervals</b> algorithm from Q1, and then the free time is simply the <b>gaps between "
    "consecutive merged intervals</b> (not before the first or after the last, since those are unbounded, "
    "not \u201cfree time\u201d in the useful sense).", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "This is a beautiful example of <b>reducing a seemingly new problem to one you already solved</b> \u2014 "
    "a hugely valuable interview skill. Once everything is merged into company-wide \u201cbusy blocks\u201d, "
    "any time not inside a busy block is, by definition, free for everyone.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def employeeFreeTime(schedule):
    intervals = []
    for emp in schedule:
        intervals.extend(emp)
    intervals.sort(key=lambda x: x[0])

    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])

    free = []
    for i in range(1, len(merged)):
        free.append([merged[i - 1][1], merged[i][0]])
    return free""", title="employee_free_time.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n log n) where n is the total number of intervals across all employees. "
               "Space: O(n).", "BodyText2"))
story.append(callout_box([
    "Always notice when a \u201cnew-looking\u201d hard problem is secretly two easy problems stapled together "
    "(here: Merge Intervals + \u201cfind the gaps\u201d). Naming the sub-problems out loud in an interview is "
    "a great way to show structured thinking."
], kind="tip"))
story.append(PageBreak())

# ---------------- Q9 Partition Labels ----------------
story += problem_header(9, "Partition Labels", "Medium", "Implicit intervals via \u201clast occurrence\u201d")
story.append(P(
    "You are given a string. Split it into as many parts as possible so that each letter appears in at most "
    "one part. Return the list of part lengths.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("Input: \"ababcbacadefegdehijhklij\" &rarr; Output: [9, 7, 8] "
               "(parts: \"ababcbaca\", \"defegde\", \"hijhklij\").", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "This problem hides an interval problem inside a string! For each character, its \u201cinterval\u201d is "
    "[first occurrence, last occurrence]. We're really doing <b>Merge Intervals</b> on these implicit "
    "character spans, but we never need to build them explicitly: track the last index of each character, "
    "then scan left to right keeping a running \u201cfarthest last-occurrence seen so far\u201d. When the "
    "current index reaches that farthest point, you've found a valid partition boundary.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "Cutting as early as the data allows is always optimal here because any wider cut can always be split "
    "further later, but you're never ALLOWED to cut earlier than a character's last occurrence (that would "
    "split that character into two parts, which breaks the rule). So \u201ccut the moment you're legally "
    "allowed to\u201d maximises the number of partitions \u2014 the same greedy flavour as interval merging.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def partitionLabels(s):
    last = {ch: i for i, ch in enumerate(s)}   # last occurrence of each char
    result = []
    start = end = 0

    for i, ch in enumerate(s):
        end = max(end, last[ch])
        if i == end:                 # reached the farthest boundary
            result.append(end - start + 1)
            start = i + 1

    return result""", title="partition_labels.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n). Space: O(1) extra (at most 26 letters in the map).", "BodyText2"))
story.append(callout_box([
    "Learning to spot \u201cinvisible intervals\u201d (first/last occurrence, earliest/latest valid position, "
    "etc.) hiding inside array or string problems is one of the highest-leverage pattern-recognition skills "
    "in all of DSA."
], kind="tip"))
story.append(PageBreak())

# ---------------- Q10 Minimum Platforms ----------------
story += problem_header(10, "Minimum Platforms Needed", "Medium", "Two-pointer sweep on sorted arrival/departure")
story.append(P(
    "(A classic GFG/railway-scheduling question, functionally identical to Meeting Rooms II.) Given arrival "
    "and departure times of trains, find the minimum number of platforms needed so that no train waits.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("arr = [900, 940, 950, 1100, 1500, 1800], dep = [910, 1200, 1120, 1130, 1900, 2000] "
               "&rarr; Output: 3", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "Identical idea to the sweep-line version of Meeting Rooms II: sort arrivals and departures separately. "
    "Walk through time; every arrival before the next unprocessed departure needs a new platform (+1), every "
    "departure that happens first frees a platform (\u22121). Track the running count's maximum.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def findMinPlatforms(arr, dep):
    arr = sorted(arr)
    dep = sorted(dep)
    n = len(arr)

    platforms_needed = 0
    max_platforms = 0
    i = j = 0

    while i < n and j < n:
        if arr[i] <= dep[j]:
            platforms_needed += 1
            i += 1
        else:
            platforms_needed -= 1
            j += 1
        max_platforms = max(max_platforms, platforms_needed)

    return max_platforms""", title="min_platforms.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n log n). Space: O(1) extra (excluding sort).", "BodyText2"))
story.append(callout_box([
    "If you've solved Meeting Rooms II, this problem is free marks \u2014 recognise it instantly as the same "
    "pattern with trains instead of meetings. Interviewers sometimes rename problems specifically to see if "
    "you notice the underlying pattern."
], kind="tip"))
story.append(PageBreak())

# ---------------- Q11 Car Pooling ----------------
story += problem_header(11, "Car Pooling", "Medium", "Difference array / sweep over passenger deltas")
story.append(P(
    "A car has a fixed capacity. You're given trips as <font face=\"Courier\">[passengers, from, to]</font>. "
    "Determine if it's possible to pick up and drop off all passengers without ever exceeding capacity.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("trips = [[2,1,5],[3,3,7]], capacity = 4 &rarr; Output: false "
               "(between location 3 and 5, both trips overlap: 2+3=5 &gt; 4).", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "This is the numeric cousin of Meeting Rooms II / Minimum Platforms, but instead of counting +1/\u22121 "
    "meetings, you're summing +passengers/\u2212passengers. Build a <b>difference array</b>: at each "
    "<font face=\"Courier\">from</font> add passengers, at each <font face=\"Courier\">to</font> subtract "
    "passengers. Then take a running prefix sum across all locations \u2014 if it ever exceeds capacity, "
    "return false.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "There's no real \u201cchoice\u201d being made \u2014 this is more of a direct simulation \u2014 but it "
    "belongs in this guide because it's the natural generalisation of the overlap-counting pattern "
    "(Meeting Rooms II) once the events carry a <b>weight</b> instead of just existing or not.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def carPooling(trips, capacity):
    changes = [0] * 1001           # locations are bounded 0..1000
    for passengers, start, end in trips:
        changes[start] += passengers
        changes[end]   -= passengers

    current = 0
    for change in changes:
        current += change
        if current > capacity:
            return False
    return True""", title="car_pooling.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n + max_location). Space: O(max_location).", "BodyText2"))
story.append(callout_box([
    "This \u201cdifference array\u201d trick (add at start, subtract at end, then prefix-sum) is one of the "
    "most reusable tricks in this whole guide \u2014 it also solves \u201cCorporate Flight Bookings\u201d and "
    "any \u201capply range updates, then read final values\u201d style problem in O(n) instead of O(n \u00d7 "
    "range length)."
], kind="tip", heading="TIP \u2014 The difference-array super-trick"))
story.append(PageBreak())

# ---------------- Q12 Jump Game ----------------
story += problem_header(12, "Jump Game", "Medium", "Greedy farthest-reachable tracking")
story.append(P(
    "Given an array where each element is the maximum jump length from that position, determine if you can "
    "reach the last index starting from index 0.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("Input: [2,3,1,1,4] &rarr; Output: true. Input: [3,2,1,0,4] &rarr; Output: false.", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "Track <font face=\"Courier\">max_reach</font>, the farthest index reachable so far. Walk through the "
    "array; if the current index ever exceeds <font face=\"Courier\">max_reach</font>, you're stuck "
    "\u2014 return false. Otherwise keep updating "
    "<font face=\"Courier\">max_reach = max(max_reach, i + nums[i])</font>.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "You never need to know <i>which exact sequence</i> of jumps gets you somewhere \u2014 only the single "
    "number \u201cfarthest point reachable so far\u201d matters, because any index within that reach is "
    "achievable by <i>some</i> combination of earlier jumps. This collapses an exponential search space "
    "(\u201ctry every jump length\u201d) into one linear pass.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def canJump(nums):
    max_reach = 0
    for i, n in enumerate(nums):
        if i > max_reach:
            return False
        max_reach = max(max_reach, i + n)
    return True""", title="jump_game.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n). Space: O(1).", "BodyText2"))
story.append(callout_box([
    "Many beginners try to brute-force this with recursion/backtracking over every jump choice (exponential "
    "time) before realising a single running maximum replaces the entire search. If you catch yourself "
    "writing recursion for a reachability question, pause and ask \u201ccan one running number replace the "
    "whole search?\u201d"
], kind="warn"))
story.append(PageBreak())

# ---------------- Q13 Jump Game II ----------------
story += problem_header(13, "Jump Game II", "Medium", "Greedy BFS-like level expansion")
story.append(P(
    "Same setup as Jump Game, but now you must return the <b>minimum number of jumps</b> to reach the last "
    "index (guaranteed reachable).", "BodyText2"))
story.append(section_label("Example"))
story.append(P("Input: [2,3,1,1,4] &rarr; Output: 2 (jump 1 step from index 0 to 1, then 3 steps to the last "
               "index).", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "Think of it like BFS \u201clevel by level\u201d, but without an explicit queue. "
    "<font face=\"Courier\">current_end</font> marks the farthest point reachable using the jumps counted "
    "so far (\u201ccurrent level\u201d). As you scan, keep extending "
    "<font face=\"Courier\">farthest</font> (the best possible next level). The moment "
    "<font face=\"Courier\">i</font> reaches <font face=\"Courier\">current_end</font>, you're forced to "
    "take one more jump, so increment the counter and set "
    "<font face=\"Courier\">current_end = farthest</font>.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "This greedily explores the maximum possible frontier at every \u201cjump level\u201d, exactly like BFS "
    "explores every node at the current distance before moving to the next distance \u2014 which is "
    "precisely why BFS (and this greedy version of it) always finds the <b>shortest</b> path/number of "
    "jumps.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def jump(nums):
    jumps = 0
    current_end = 0
    farthest = 0

    for i in range(len(nums) - 1):     # don't need to jump FROM the last index
        farthest = max(farthest, i + nums[i])
        if i == current_end:
            jumps += 1
            current_end = farthest

    return jumps""", title="jump_game_2.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n). Space: O(1).", "BodyText2"))
story.append(callout_box([
    "Notice the loop stops at <font face=\"Courier\">len(nums) - 1</font>, not "
    "<font face=\"Courier\">len(nums)</font> \u2014 we never need to \u201cjump from\u201d the final index. "
    "An off-by-one here is a very common submission error."
], kind="warn"))
story.append(PageBreak())

# ---------------- Q14 Gas Station ----------------
story += problem_header(14, "Gas Station", "Medium", "Prefix-sum greedy restart")
story.append(P(
    "There are <font face=\"Courier\">n</font> gas stations in a circle. You're given "
    "<font face=\"Courier\">gas[i]</font> (fuel available at station i) and "
    "<font face=\"Courier\">cost[i]</font> (fuel needed to travel from station i to i+1). Find the starting "
    "station index from which you can complete the full circuit, or return -1 if impossible.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("gas = [1,2,3,4,5], cost = [3,4,5,1,2] &rarr; Output: 3.", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "Two key observations combine into one clean pass: (1) a solution exists if and only if "
    "<font face=\"Courier\">sum(gas) &gt;= sum(cost)</font>; (2) if your running fuel tank (starting from "
    "some candidate station) ever goes negative at station <font face=\"Courier\">i</font>, then <b>no "
    "station between your current start and i</b> could have been a valid starting point either \u2014 so "
    "you can safely jump your candidate start straight to <font face=\"Courier\">i+1</font> and reset the "
    "tank to 0.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "If starting at station <font face=\"Courier\">S</font> and you run out of fuel at station "
    "<font face=\"Courier\">i</font> (i.e. your cumulative tank first goes negative there), then for any "
    "station <font face=\"Courier\">k</font> between S and i, starting at k would arrive at i with an "
    "equal-or-smaller tank (since you'd be skipping some of the surplus fuel collected between S and k). "
    "So none of those intermediate stations can possibly work either \u2014 they can all be eliminated in "
    "one shot, which is what lets a single linear pass solve the whole problem.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def canCompleteCircuit(gas, cost):
    total_tank = 0
    current_tank = 0
    start = 0

    for i in range(len(gas)):
        diff = gas[i] - cost[i]
        total_tank += diff
        current_tank += diff

        if current_tank < 0:      # can't reach i+1 from "start"
            start = i + 1         # every station between old start and i is eliminated
            current_tank = 0

    return start if total_tank >= 0 else -1""", title="gas_station.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n). Space: O(1).", "BodyText2"))
story.append(callout_box([
    "This is widely considered one of the trickiest \u201ceasy-looking\u201d greedy problems precisely "
    "because the correctness argument (\u201celiminating a whole range of candidates in one shot\u201d) isn't "
    "obvious at first glance. Practice explaining observation (2) above out loud until it feels natural "
    "\u2014 it's a favourite \u201cwhy does this work?\u201d follow-up question."
], kind="tip"))
story.append(PageBreak())

# ---------------- Q15 Task Scheduler ----------------
story += problem_header(15, "Task Scheduler", "Medium", "Greedy formula using max-frequency slot counting")
story.append(P(
    "Given a list of CPU tasks (as characters) and a cooldown <font face=\"Courier\">n</font> "
    "(the same task must be separated by at least n intervals), find the minimum total time (including "
    "idle slots) needed to finish all tasks.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("tasks = [A,A,A,B,B,B], n=2 &rarr; Output: 8 "
               "(e.g. A B idle A B idle A B).", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "Find the task with the <b>highest frequency</b>, call it <font face=\"Courier\">f_max</font>, and "
    "count how many <b>different</b> tasks also share that maximum frequency, call it "
    "<font face=\"Courier\">count_max</font>. Picture laying out "
    "<font face=\"Courier\">(f_max - 1)</font> \u201cchunks\u201d of size <font face=\"Courier\">(n + 1)</font>, "
    "with the final partial chunk holding exactly <font face=\"Courier\">count_max</font> tasks. Then the "
    "answer is the larger of this theoretical frame size and simply the total number of tasks (because if "
    "there are enough different tasks to fill every idle slot, there's no idle time at all).", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "Greedily scheduling the most frequent task as early and as spread-out as possible in each "
    "<font face=\"Courier\">(n+1)</font>-sized window guarantees the fewest idle slots, because any other "
    "ordering can only push the bottleneck task's repeats <b>further apart</b>, never closer \u2014 so this "
    "arrangement is provably the tightest packing possible.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""from collections import Counter

def leastInterval(tasks, n):
    freq = Counter(tasks)
    f_max = max(freq.values())
    count_max = sum(1 for v in freq.values() if v == f_max)

    frame_size = (f_max - 1) * (n + 1) + count_max
    return max(frame_size, len(tasks))""", title="task_scheduler.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n) where n is the number of tasks (building the frequency map). Space: O(1) "
               "(at most 26 distinct uppercase task types).", "BodyText2"))
story.append(callout_box([
    "This formula feels like \u201cmagic\u201d the first time you see it. Don't just memorise it \u2014 draw "
    "the chunks on paper: <font face=\"Courier\">(f_max-1)</font> full blocks of size "
    "<font face=\"Courier\">(n+1)</font>, plus one final partial block of exactly "
    "<font face=\"Courier\">count_max</font> tasks. Once you've drawn it once, you'll never forget the "
    "formula."
], kind="tip"))
story.append(callout_box([
    "<font face=\"Courier\">max(frame_size, len(tasks))</font> is essential and frequently forgotten: when "
    "there are many different task types, the frame-based formula can under-count, because there are already "
    "enough distinct tasks to fill every gap with zero idling \u2014 in that case the real answer is simply "
    "the total task count."
], kind="warn"))
story.append(PageBreak())

# ---------------- Q16 Queue Reconstruction ----------------
story += problem_header(16, "Queue Reconstruction by Height", "Medium", "Sort smartly, then greedy-insert")
story.append(P(
    "Each person is given as <font face=\"Courier\">[height, k]</font> where k is the number of people in "
    "front of them who are taller or equal in height. Reconstruct the queue.", "BodyText2"))
story.append(section_label("Example"))
story.append(P("Input: [[7,0],[4,4],[7,1],[5,0],[6,1],[5,2]] "
               "&rarr; Output: [[5,0],[7,0],[5,2],[6,1],[4,4],[7,1]]", "BodyText2"))
story.append(section_label("Thought Process"))
story.append(P(
    "Sort people by <b>height descending</b>, and for equal heights by <b>k ascending</b>. Then insert each "
    "person into the result list at index <font face=\"Courier\">k</font> (using Python's "
    "<font face=\"Courier\">list.insert</font>). Because taller people are placed first, inserting a shorter "
    "person later at position k never disturbs the relative \u201ctaller-or-equal count\u201d of anyone "
    "already placed \u2014 shorter people don't count towards a taller person's k.", "BodyText2"))
story.append(section_label("Why Greedy Works"))
story.append(P(
    "Once all taller-or-equal people are placed, a shorter person's own k value tells you <i>exactly</i> "
    "how many of those taller people must stand in front of them \u2014 so slotting them directly at index k "
    "is always correct and never needs revisiting, because future (even shorter) insertions can't possibly "
    "affect this person's k count either.", "BodyText2"))
story.append(section_label("Code (Python)"))
story.append(code_block(
"""def reconstructQueue(people):
    # tallest first; for ties, smaller k first
    people.sort(key=lambda p: (-p[0], p[1]))
    result = []
    for person in people:
        result.insert(person[1], person)
    return result""", title="queue_reconstruction.py"))
story.append(section_label("Complexity"))
story.append(P("Time: O(n log n) for sorting + O(n\u00b2) worst case for repeated list insertions "
               "(each insert can shift up to n elements). Space: O(n).", "BodyText2"))
story.append(callout_box([
    "This is a great example of a problem where the <b>sort key itself</b> is the entire trick \u2014 the "
    "insertion step is almost mechanical once you see why \u201ctallest first, ties broken by smaller "
    "k\u201d is the only ordering that makes every future insertion \u201csafe\u201d."
], kind="tip"))

story.append(PageBreak())

# ======================================================================
# SECTION 7: MASTER TECHNIQUE TOOLBOX
# ======================================================================
story += H1("7. Master Technique Toolbox (Tips &amp; Tricks)")

story += H3("1. The Sweep Line / Event Technique")
story.append(P(
    "Convert every interval <font face=\"Courier\">[s, e]</font> into two \u201cevents\u201d: "
    "<font face=\"Courier\">(s, +1)</font> and <font face=\"Courier\">(e, \u22121)</font>. Sort all events by "
    "time (and process \u22121 before +1 at the exact same timestamp if touching intervals should NOT count "
    "as overlapping). Walk through, maintaining a running sum \u2014 its maximum value across the whole scan "
    "is the <b>maximum number of intervals overlapping at any single instant</b>.", "BodyText2"))
story.append(code_block(
"""def max_overlap(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 1))
        events.append((e, -1))
    events.sort(key=lambda x: (x[0], x[1]))  # -1 before +1 on ties = touching doesn't overlap

    count = max_count = 0
    for time, delta in events:
        count += delta
        max_count = max(max_count, count)
    return max_count""", title="sweep_line_template.py"))

story += H3("2. The Difference Array Trick")
story.append(P(
    "When you need to apply <b>range updates</b> (add a value to every index between L and R) many times, "
    "and only need the <b>final</b> array afterwards, don't loop over the whole range each time (that's "
    "O(n) per update). Instead: <font face=\"Courier\">diff[L] += val</font>, "
    "<font face=\"Courier\">diff[R+1] -= val</font>, then take one prefix sum at the end. "
    "Each update becomes O(1), and the final reconstruction is a single O(n) pass.", "BodyText2"))

story += H3("3. Min-Heap for \u201cEarliest Available Slot\u201d Problems")
story.append(P(
    "Whenever a problem asks you to repeatedly find/reuse \u201cwhichever resource becomes free "
    "soonest\u201d (rooms, servers, machines, elevators), a min-heap keyed on \u201cfree-at time\u201d is "
    "almost always the right data structure \u2014 it gives you the earliest-freeing resource in O(log k) "
    "time where k is the number of resources currently in use.", "BodyText2"))

story += H3("4. Exchange Argument Sentence Template")
story.append(P(
    "To justify any interval greedy rule in an interview, fill in this template: <i>\u201cSuppose an optimal "
    "solution made a different choice at this step. I can always swap that choice for mine without making "
    "the solution worse, because ___. Therefore my choice is at least as good, so greedy is safe.\u201d</i>",
    "BodyText2"))

story += H3("5. Recognise \u201cHidden Intervals\u201d")
story.append(P(
    "Not every interval problem shows you <font face=\"Courier\">[start, end]</font> pairs directly. Watch "
    "for: first/last occurrence of a character in a string (Partition Labels), a value's valid range before "
    "it conflicts with another value, a time window implied by \u201cwithin k days of\u201d, or coordinates "
    "on a line representing reach/visibility.", "BodyText2"))

story += H3("6. Sorting With Custom Tie-Breakers in Python")
story.append(code_block(
"""# Sort by end ascending, tie-break by start ascending
intervals.sort(key=lambda x: (x[1], x[0]))

# Sort by height descending, tie-break by k ascending (Queue Reconstruction)
people.sort(key=lambda p: (-p[0], p[1]))""", title="sort_tiebreakers.py"))

story += H3("7. Always Handle the Empty / Single-Element Edge Case First")
story.append(P(
    "Interval problems frequently crash on an empty input list or a single interval because code assumes "
    "\u201cat least 2 elements\u201d when initialising a \u201ccurrent/previous\u201d tracker. Add an early "
    "return for <font face=\"Courier\">len(intervals) <= 1</font> wherever relevant.", "BodyText2"))

story.append(PageBreak())

# ======================================================================
# SECTION 8: CONFUSION POINTS
# ======================================================================
story += H1("8. Confusion Points \u2014 Clarified in Depth")

story += H3("Confusion 1: \u201cDo I sort by start or by end?\u201d")
story.append(P(
    "Resolved fully in Section 5 \u2014 but as a quick mental shortcut: if the task is to <b>combine/group</b> "
    "things that touch (Merge Intervals, Employee Free Time), sort by <b>start</b>. If the task is to "
    "<b>select the maximum count</b> of non-conflicting things or minimise removals (Activity Selection, "
    "Non-overlapping Intervals, Min Arrows), sort by <b>end</b>.", "BodyText2"))

story += H3("Confusion 2: \u201cIs greedy always slower/faster than DP?\u201d")
story.append(P(
    "Greedy is virtually always <b>faster</b> than DP when both apply to the same problem (O(n log n) vs. "
    "O(n\u00b2) or worse), precisely <b>because</b> greedy throws away information DP would have kept around "
    "\u201cjust in case\u201d. The trade-off is that greedy only works when you can prove it's safe to throw "
    "that information away.", "BodyText2"))

story += H3("Confusion 3: \u201cWhy do interval problems always seem to need O(n log n)?\u201d")
story.append(P(
    "Because almost every interval greedy algorithm has exactly two phases: <b>sort</b> (O(n log n)) and "
    "<b>single linear scan</b> (O(n)). The sort dominates, so the overall complexity is O(n log n). If a "
    "problem gives you pre-sorted input (like Insert Interval or Interval List Intersections), the "
    "complexity drops to pure O(n) because you skip the sort entirely.", "BodyText2"))

story += H3("Confusion 4: Inclusive vs. exclusive interval endpoints")
story.append(P(
    "Revisit this every single time you see a new problem. <font face=\"Courier\">[1,5]</font> and "
    "<font face=\"Courier\">[5,10]</font>: touching at 5. Does the problem treat position 5 as belonging to "
    "both, neither, or does it simply not matter because positions are continuous moments in time rather "
    "than discrete integers? Re-read the problem's own worked examples \u2014 never assume based on a "
    "similar problem you solved before.", "BodyText2"))

story += H3("Confusion 5: \u201cGreedy failed on my custom test \u2014 does that mean greedy never works "
             "here?\u201d")
story.append(P(
    "Not necessarily \u2014 it might mean your <b>greedy rule</b> was wrong, not that greedy as a paradigm "
    "is wrong for this problem. For example, in Activity Selection, sorting by <b>duration</b> "
    "(shortest activity first) feels intuitive but is actually <b>incorrect</b> \u2014 a short activity in "
    "the middle of the timeline can block more future activities than a short activity sorted by end time "
    "would. Always double check you picked the provably correct greedy rule (Section 3), not just any "
    "rule that feels greedy.", "BodyText2"))
story.append(callout_box([
    "Counter-example proving \u201csort by shortest duration first\u201d is wrong for Activity Selection: "
    "intervals [1,10] (duration 9), [2,3] (duration 1), [4,5] (duration 1), [6,7] (duration 1). Sorting by "
    "shortest duration might greedily grab [2,3] first \u2014 fine so far \u2014 but a careless "
    "duration-based rule can still end up blocked compared to simply sorting by end time, which reliably "
    "gets [2,3], [4,5], [6,7] all three. Always default to <b>sort by end time</b> for this family of "
    "problems; it's the one rule with a solid proof behind it."
], kind="warn", heading="COMMON CONFUSION \u2014 \u201cSort by shortest duration\u201d seems intuitive but isn't proven safe"))

story += H3("Confusion 6: Meeting Rooms I vs. II \u2014 when do I need a heap?")
story.append(P(
    "Meeting Rooms I only asks \u201ccan one person attend everything?\u201d \u2014 a yes/no overlap check, no "
    "heap needed. Meeting Rooms II asks \u201chow many simultaneous resources (rooms) are ever needed at "
    "once?\u201d \u2014 this requires <b>counting concurrent overlaps</b>, which is what the heap (or "
    "sweep-line two-pointer) is for. If a problem only asks yes/no, you're almost certainly over-engineering "
    "if you reach for a heap.", "BodyText2"))

story += H3("Confusion 7: \u201cWhy does Gas Station's greedy restart work? Isn't that cheating?\u201d")
story.append(P(
    "It feels like cheating because we \u201cskip checking\u201d several candidate starting points at once. "
    "But Section 6, Q14 proves formally why all of those skipped candidates are guaranteed to fail too "
    "\u2014 so skipping them isn't an approximation, it's a mathematically justified elimination.", "BodyText2"))

story.append(PageBreak())

# ======================================================================
# SECTION 9: COMPLEXITY CHEAT SHEET
# ======================================================================
story += H1("9. Complexity Cheat Sheet")

complexity_table = info_table([
    ["Problem", "Time", "Space", "Core Data Structure"],
    ["Merge Intervals", "O(n log n)", "O(n)", "Sorted array, running block"],
    ["Insert Interval", "O(n)", "O(n)", "None (3-zone scan)"],
    ["Non-overlapping Intervals", "O(n log n)", "O(1)", "Sorted array"],
    ["Meeting Rooms I", "O(n log n)", "O(1)", "Sorted array"],
    ["Meeting Rooms II", "O(n log n)", "O(n)", "Min-heap (or 2 sorted arrays)"],
    ["Min Arrows to Burst Balloons", "O(n log n)", "O(1)", "Sorted array"],
    ["Interval List Intersections", "O(m+n)", "O(1)*", "Two pointers"],
    ["Employee Free Time", "O(n log n)", "O(n)", "Sorted/merged array"],
    ["Partition Labels", "O(n)", "O(1)", "Hash map (last index)"],
    ["Minimum Platforms", "O(n log n)", "O(1)", "Two sorted arrays"],
    ["Car Pooling", "O(n + range)", "O(range)", "Difference array"],
    ["Jump Game", "O(n)", "O(1)", "Running max"],
    ["Jump Game II", "O(n)", "O(1)", "Running max (BFS-style levels)"],
    ["Gas Station", "O(n)", "O(1)", "Running prefix sum"],
    ["Task Scheduler", "O(n)", "O(1)", "Frequency map"],
    ["Queue Reconstruction", "O(n log n) + O(n\u00b2)", "O(n)", "Sorted list + insertion"],
], col_widths=[52 * mm, 32 * mm, 22 * mm, 56 * mm])
story.append(complexity_table)
story.append(Spacer(1, 6))
story.append(P("*excluding space for the output list.", "SmallCaption"))

story.append(PageBreak())

# ======================================================================
# SECTION 10: PATTERN RECOGNITION
# ======================================================================
story += H1("10. Pattern Recognition: Spotting a Greedy-Interval Question")

story.append(P(
    "Train yourself to recognise these phrases in a problem statement \u2014 they're strong signals of the "
    "pattern being tested.", "BodyText2"))

pattern_table = info_table([
    ["Phrase in the Problem", "Likely Pattern"],
    ["\u201cmerge all overlapping\u2026\u201d", "Merge Intervals (sort by start)"],
    ["\u201cmaximum number of non-overlapping\u2026\u201d / \u201cminimum removals so none overlap\u201d", "Activity Selection (sort by end)"],
    ["\u201ccan one person attend all\u2026\u201d", "Simple overlap check (Meeting Rooms I)"],
    ["\u201cminimum number of rooms / resources / platforms\u2026\u201d", "Max concurrent overlap (heap or sweep line)"],
    ["\u201cminimum arrows / minimum points to cover all ranges\u2026\u201d", "Activity Selection variant (sort by end)"],
    ["\u201cfree time common to all\u2026\u201d", "Merge Intervals + gap finding"],
    ["\u201ceach letter/element appears in only one part\u2026\u201d", "Hidden intervals via first/last occurrence"],
    ["\u201ccapacity must never be exceeded\u2026\u201d with ranges", "Difference array / sweep line with weights"],
    ["\u201cmaximum index reachable\u2026\u201d / \u201cminimum jumps\u2026\u201d", "Running max reach (Jump Game family)"],
    ["\u201ccircular route, enough fuel\u2026\u201d", "Prefix sum + greedy restart (Gas Station)"],
    ["\u201ccooldown between repeated tasks\u2026\u201d", "Frequency-based greedy slot formula"],
], col_widths=[95 * mm, 67 * mm])
story.append(pattern_table)

story.append(Spacer(1, 10))
story.append(callout_box([
    "When you spot one of these phrases, immediately ask yourself the Section 5 sentence: <i>\u201cI am "
    "sorting by ___ because I need to know ___ as I scan.\u201d</i> Nine times out of ten, that single "
    "sentence alone leads you straight to working code."
], kind="tip"))

story.append(PageBreak())

# ======================================================================
# SECTION 11: PRACTICE LIST + CHECKLIST
# ======================================================================
story += H1("11. Practice List &amp; Final Interview Checklist")

story += H3("Recommended Practice Order (Easy &rarr; Hard)")
story += numbered([
    "Meeting Rooms I (warm-up overlap check)",
    "Merge Intervals",
    "Insert Interval",
    "Non-overlapping Intervals",
    "Minimum Number of Arrows to Burst Balloons",
    "Partition Labels",
    "Jump Game",
    "Meeting Rooms II",
    "Minimum Platforms Needed",
    "Interval List Intersections",
    "Jump Game II",
    "Gas Station",
    "Car Pooling",
    "Task Scheduler",
    "Queue Reconstruction by Height",
    "Employee Free Time",
])

story += H3("Final Checklist \u2014 Run Through This Before Every Interval Problem")
story += bullets([
    "Have I re-read the examples to know whether <b>touching endpoints overlap or not</b>?",
    "Have I decided <b>sort by start or sort by end</b>, and can I say <i>why</i> in one sentence?",
    "Have I checked the <b>empty list</b> and <b>single element</b> edge cases?",
    "Can I explain <b>why the greedy choice can't be beaten</b> by some other choice (exchange argument)?",
    "Is my overlap/merge condition using the <b>correct comparison operator</b> (&lt; vs &lt;=)?",
    "Have I double-checked the <b>time complexity</b> matches the expected O(n log n) or O(n)?",
    "If counting concurrent overlaps, have I considered whether a <b>heap or two-pointer sweep</b> is cleaner?",
    "Have I considered whether this is secretly a <b>\u201chidden interval\u201d</b> problem (first/last "
    "occurrence, reach, range)?",
])

story.append(Spacer(1, 16))
story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#C9D4E3"), spaceAfter=10))
story.append(P("\u201cGreedy algorithms don't guess \u2014 they commit, because they can prove, step by "
               "step, that no better choice ever existed.\u201d", "FooterQuote"))
story.append(Spacer(1, 6))
story.append(P("End of Guide \u2014 Keep practising, keep proving your greedy choices, and the pattern "
               "becomes second nature.", "FooterQuote"))

# ----------------------------------------------------------------------
doc.build(story)
print("PDF built successfully.")

PDF built successfully.
